# Smoke run (20 questions) — rag-evidence-attribution-bench

Thin wrapper: every stage is one CLI call; all logic lives in `src/rag_evidence/`.

**Prerequisites**
1. Build the bundle locally: `uv run python scripts/make_colab_bundle.py`
2. Upload `reab_bundle.zip` to Google Drive at `MyDrive/reab/reab_bundle.zip`
3. Runtime → Change runtime type → **GPU** (T4 is enough)

**Estimated wall-clock on a T4 (estimate, not a measurement): ~25–45 min** including
model download. Every cell is idempotent — after a disconnect, Runtime → Run all;
`--resume` skips completed samples (checkpoints live on Drive).

Output: `results_smoke_<stamp>.zip` on Drive under `MyDrive/reab/results/export/` —
bring it home and run `python -m rag_evidence.cli import-results <zip> --config configs/smoke.yaml`.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU runtime — Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("torch:", torch.__version__)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
BUNDLE_ZIP = '/content/drive/MyDrive/reab/reab_bundle.zip'
DRIVE_RESULTS = '/content/drive/MyDrive/reab/results'
assert os.path.exists(BUNDLE_ZIP), f'upload the bundle to {BUNDLE_ZIP} first'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
# checkpoints go to Drive (survive disconnects); code runs from fast local disk
os.environ['RAG_EVIDENCE_RESULTS_RAW'] = f'{DRIVE_RESULTS}/raw'
os.environ['RAG_EVIDENCE_RESULTS_DERIVED'] = f'{DRIVE_RESULTS}/derived'
os.environ['HF_HOME'] = '/content/hf_cache'

In [ ]:
!rm -rf /content/reab && mkdir -p /content/reab
!unzip -q "$BUNDLE_ZIP" -d /content/reab
%cd /content/reab
# pin the libraries that change numbers to the versions the repo was locked with;
# torch is deliberately NOT pinned (Colab's preinstalled CUDA torch stays)
!pip install -q "transformers==5.14.1" "tokenizers==0.22.2" "accelerate>=1.0" "bitsandbytes>=0.49" "datasets>=3.0" "sentence-transformers>=5.0"
!pip install -q -e ".[ml,gpu]"
# seed Drive results with the repo's committed results (no-clobber) so GPU stages
# append to the same runs and import-results never sees raw conflicts
!mkdir -p "$DRIVE_RESULTS" && cp -rn results/raw "$DRIVE_RESULTS/" 2>/dev/null; cp -rn results/derived "$DRIVE_RESULTS/" 2>/dev/null; true

In [ ]:
!python -m rag_evidence.cli --version
!python -m rag_evidence.cli status --config configs/smoke.yaml

`data prepare` is NOT needed here: the bundle ships the prepared splits and the
committed fingerprint manifest; every stage re-verifies fingerprints at startup.

In [ ]:
!python -m rag_evidence.cli generate --config configs/smoke.yaml --resume

In [ ]:
!python -m rag_evidence.cli attribute --method citations     --config configs/smoke.yaml --resume
!python -m rag_evidence.cli attribute --method embedding     --config configs/smoke.yaml --resume
# longest cell: leave-one-out = 11 teacher-forced passes/sample/mode + faithfulness
!python -m rag_evidence.cli attribute --method leave_one_out --config configs/smoke.yaml --resume

In [ ]:
%%bash
for m in control_random control_retrieval control_lexical control_length control_shuffled; do
  python -m rag_evidence.cli attribute --method $m --config configs/smoke.yaml --resume
done

In [ ]:
# OPTIONAL — ContextCite adapter (surrogate-model attribution, ~64 ablations/sample).
# Note: it attributes its OWN regeneration under its own prompt template, not the
# stored generation-run answer (see MODEL_CARD.md). Mode B only.
!pip install -q context-cite
!python -m rag_evidence.cli attribute --method contextcite --config configs/smoke.yaml --resume

In [ ]:
!python -m rag_evidence.cli evaluate --config configs/smoke.yaml
!python -m rag_evidence.cli report   --config configs/smoke.yaml

In [ ]:
!python -m rag_evidence.cli export --config configs/smoke.yaml
import glob
from google.colab import files
latest = sorted(glob.glob(f"{DRIVE_RESULTS}/../results/export/*.zip") + glob.glob('results/export/*.zip'))
print('export zips:', latest)
if latest:
    files.download(latest[-1])